#### Code pour faire marcher PySpark chez moi 

In [3]:
import os, subprocess, shutil
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ.get("PATH", "")
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"
print("winutils:", os.path.exists(r"C:\hadoop\bin\winutils.exe"))
print("hadoop.dll:", os.path.exists(r"C:\hadoop\bin\hadoop.dll"))

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot" 
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + ";" + os.environ.get("PATH", "")
print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("java in PATH:", shutil.which("java"))
print(subprocess.run(["java","-version"], capture_output=True, text=True).stderr)

winutils: True
hadoop.dll: True
JAVA_HOME: C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot
java in PATH: C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot\bin\java.EXE
openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment Temurin-17.0.18+8 (build 17.0.18+8)
OpenJDK 64-Bit Server VM Temurin-17.0.18+8 (build 17.0.18+8, mixed mode, sharing)



In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .config("spark.sql.ansi.enabled", "false")
    .getOrCreate()
)

import pyspark.pandas as ps
ps.set_option("compute.fail_on_ansi_mode", False)

print("ansi =", spark.conf.get("spark.sql.ansi.enabled"))

ansi = false


# Code principal tests

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyspark.pandas as ps
from pyspark.sql.functions import *

In [7]:
mission_paris = pd.read_csv(".\data\BDD_BGES\BDD_BGES_PARIS\BDD_BGES_PARIS_MISSION\MISSION_20260429.txt", sep=";", index_col="ID_MISSION")

In [11]:
mission_paris["VILLE_DEPART"].unique()

array(['Paris', 'Shanghai'], dtype=object)

In [4]:
pers_paris = pd.read_csv(".\data\BDD_BGES\BDD_BGES_PARIS\PERSONNEL_PARIS.txt", sep=";", index_col="ID_PERSONNEL")

In [5]:
pers_paris.head()

,NOM_PERSONNEL,PRENOM_PERSONNEL,DT_NAISS,VILLE_NAISS,PAYS_NAISS,NUM_SECU,IND_PAYS_NUM_TELP,NUM_TELEPHONE,NUM_VOIE,DSC_VOIE,CMPL_VOIE,CD_POSTAL,VILLE,PAYS,FONCTION_PERSONNEL,TS_CREATION_PERSONNEL,TS_MAJ_PPERSONNEL
ID_PERSONNEL,,,,,,,,,,,,,,,,,
KeyPers_Paris_1230000,Name0,FistName0,1931-07-27,Tokyo,Japan,NS000000000,NaN,+336##0014965,28,NomVoie510,NaN,#7168,Paris,France,Cadre,2009-03-23 14:28:33,2009-03-23 14:28:33
KeyPers_Paris_1230001,Name1,FistName1,1963-01-07,Melbourne,Australia,NS000000001,NaN,+336##0743025,60,NomVoie171,NaN,#9979,Paris,France,DRH,2017-05-10 07:55:58,2017-05-10 07:55:58
KeyPers_Paris_1230002,Name2,FistName2,1979-04-03,Sidney,Australia,NS000000002,NaN,+336##0098564,73,NomVoie382,NaN,#9505,Paris,France,Ingénieur Data,2002-11-23 16:12:12,2002-11-23 16:12:12
KeyPers_Paris_1230003,Name3,FistName3,1944-08-01,Montreal,Canada,NS000000003,NaN,+336##0171838,36,NomVoie323,NaN,#0725,Paris,France,Ingénieur Informaticien,1999-01-15 12:34:15,1999-01-15 12:34:15
KeyPers_Paris_1230004,Name4,FistName4,1981-01-21,Lille,France,NS000000004,NaN,+336##0570970,69,NomVoie554,NaN,#8557,Paris,France,Ingénieur Informaticien,2020-01-04 22:34:38,2020-01-04 22:34:38


#### Test ETL

Fonction readFiles : réunir tous les .txt d'un dossier en un seul dataframe. C'est peut-être débile de faire ça car y'a moyen qu'un fichier corresponde à une journée donc en fonction de comment on traite les fichiers à voir

In [11]:
def readFiles(path, indexCol):
    p = Path(path)
    files = sorted(p.glob("*.txt"))  #liste des fichiers dans le dossier
    dfs = []
    """
    for f in files:
        df = pd.read_csv(f, sep=";", index_col=indexCol)  
        df["source_file"] = f.name # optionnel si on veut garder le fichier d'où vient l'information              
        dfs.append(df)
    """

    """
    # Sinon en compréhension de liste sans la colonne source_file
    dfs = [pd.read_csv(f, sep=';', index_col=indexCol) for f in files]
    all_df = pd.concat(dfs, verify_integrity=True)  # ValueError si doublons d'index
    """

    # Avec pyspark (très lent)
    dfs = [ps.read_csv(str(f), sep=';', index_col=indexCol) for f in files]
    all_df = ps.concat(dfs) 
    tmp = all_df.reset_index()
    dups = tmp[tmp.duplicated(subset=indexCol, keep=False)][indexCol].unique().to_list()
    if dups:
        raise ValueError(f"Doublons d'{indexCol} détectés: {dups}")
    

    return all_df

In [12]:
mat_info_paris_psdf = readFiles("data/BDD_BGES/BDD_BGES_PARIS/BDD_BGES_PARIS_INFORMATIQUE", "ID_MATERIELINFO")
mat_info_paris_psdf.shape[0]

c:\Users\etien\miniconda3\envs\nf26\Lib\site-packages\pyspark\pandas\utils.py:1037: PandasAPIOnSparkAdviceWarning: `to_list` loads all data into the driver's memory. It should only be used if the resulting list is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


1527

In [13]:
mat_info_paris_sdf = mat_info_paris_psdf.to_spark(index_col="ID_MATERIELINFO")

In [14]:
mat_info_paris_sdf.show(10)

+--------------------+--------------------+-------------+----------------+-------------------+------------------+--------------------+
|     ID_MATERIELINFO|        ID_PERSONNEL|NOM_PERSONNEL|PRENOM_PERSONNEL|         DATE_ACHAT|              TYPE|              MODELE|
+--------------------+--------------------+-------------+----------------+-------------------+------------------+--------------------+
|Paris_MATERIEL_IN...|KeyPers_Paris_123...|     Name2362|    FistName2362|2026-04-29 08:17:31|PC fixe sans ecran|                   Z|
|Paris_MATERIEL_IN...|KeyPers_Paris_123...|     Name2165|    FistName2165|2026-04-29 09:42:55|        imprimante|   Laser A3 (>100kg)|
|Paris_MATERIEL_IN...|KeyPers_Paris_123...|     Name1951|    FistName1951|2026-04-29 13:58:12|PC fixe sans ecran|Precision tower 3xxx|
|Paris_MATERIEL_IN...|KeyPers_Paris_123...|      Name614|     FistName614|2026-04-29 13:19:31|      Telephone IP|                    |
|Paris_MATERIEL_IN...|KeyPers_Paris_123...|     Name295

#### Homogénéisation de la langue des fonctions

In [22]:
pers_london_psdf = readFiles("data/BDD_BGES/BDD_BGES_LONDON", "ID_PERSONNEL")
pers_london_psdf["FONCTION_PERSONNEL"].unique()

c:\Users\etien\miniconda3\envs\nf26\Lib\site-packages\pyspark\pandas\utils.py:1037: PandasAPIOnSparkAdviceWarning: `to_list` loads all data into the driver's memory. It should only be used if the resulting list is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


0     Computer Engineer
1                   HRD
2    Business Executive
3             Economist
4         Data Engineer
Name: FONCTION_PERSONNEL, dtype: object

In [20]:
pers_berlin_psdf = readFiles("data/BDD_BGES/BDD_BGES_BERLIN", "ID_PERSONNEL")
pers_berlin_psdf["FONCTION_PERSONNEL"].unique()

c:\Users\etien\miniconda3\envs\nf26\Lib\site-packages\pyspark\pandas\utils.py:1037: PandasAPIOnSparkAdviceWarning: `to_list` loads all data into the driver's memory. It should only be used if the resulting list is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


0               Ökonom
1        Führungskraft
2       Personalleiter
3    Computeringenieur
4       Dateningenieur
Name: FONCTION_PERSONNEL, dtype: object

In [23]:
pers_paris_psdf = readFiles("data/BDD_BGES/BDD_BGES_PARIS", "ID_PERSONNEL")
pers_paris_psdf["FONCTION_PERSONNEL"].unique()

c:\Users\etien\miniconda3\envs\nf26\Lib\site-packages\pyspark\pandas\utils.py:1037: PandasAPIOnSparkAdviceWarning: `to_list` loads all data into the driver's memory. It should only be used if the resulting list is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


0    Ingénieur Informaticien
1             Ingénieur Data
2                 Economiste
3                        DRH
4                      Cadre
Name: FONCTION_PERSONNEL, dtype: object

In [27]:
# Comme l'anglais est la langue de 4 sites, on homogeneise en anglais pour avoir moins d'opérations 
def clean_langue_fonction(psdf, site):
    match site:
        case "BERLIN":
            map_fonctions = {
                "Ökonom": "Economist",
                "Führungskraft": "Business Executive",
                "Personalleiter": "HRD",
                "Computeringenieur": "Computer Engineer",
                "Dateningenieur": "Data Engineer",
            }
            psdf["FONCTION_PERSONNEL"] = psdf["FONCTION_PERSONNEL"].replace(map_fonctions)
        case "PARIS":
            map_fonctions = {
                "Ingénieur Informaticien": "Computer Engineer",
                "Ingénieur Data": "Data Engineer",
                "Economiste": "Economist",
                "DRH": "HRD",
                "Cadre": "Business Executive",
            }
            psdf["FONCTION_PERSONNEL"] = psdf["FONCTION_PERSONNEL"].replace(map_fonctions)


In [28]:
clean_langue_fonction(pers_paris_psdf, "PARIS")
pers_paris_psdf["FONCTION_PERSONNEL"].unique()

0     Computer Engineer
1                   HRD
2    Business Executive
3             Economist
4         Data Engineer
Name: FONCTION_PERSONNEL, dtype: object

In [29]:
clean_langue_fonction(pers_berlin_psdf, "BERLIN")
pers_paris_psdf["FONCTION_PERSONNEL"].unique()

0     Computer Engineer
1                   HRD
2    Business Executive
3             Economist
4         Data Engineer
Name: FONCTION_PERSONNEL, dtype: object

La langue est bien mise en anglais partout

#### Homogénéisation de la langue des types de missions

In [33]:
mission_londres_psdf = readFiles("data/BDD_BGES/BDD_BGES_LONDON/BDD_BGES_LONDON_MISSION", "ID_MISSION")
mission_londres_psdf["TYPE_MISSION"].unique()

c:\Users\etien\miniconda3\envs\nf26\Lib\site-packages\pyspark\pandas\utils.py:1037: PandasAPIOnSparkAdviceWarning: `to_list` loads all data into the driver's memory. It should only be used if the resulting list is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


0       Business Meeting
1             Conference
2           Team Meeting
3            Development
4    Vocational Training
Name: TYPE_MISSION, dtype: object

In [34]:
mission_berlin_psdf = readFiles("data/BDD_BGES/BDD_BGES_BERLIN/BDD_BGES_BERLIN_MISSION", "ID_MISSION")
mission_berlin_psdf["TYPE_MISSION"].unique()

c:\Users\etien\miniconda3\envs\nf26\Lib\site-packages\pyspark\pandas\utils.py:1037: PandasAPIOnSparkAdviceWarning: `to_list` loads all data into the driver's memory. It should only be used if the resulting list is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


0    Geschäftstreffen
1           Konferenz
2            Schulung
3             Meeting
4         Entwicklung
Name: TYPE_MISSION, dtype: object

In [31]:
mission_paris_psdf = readFiles("data/BDD_BGES/BDD_BGES_PARIS/BDD_BGES_PARIS_MISSION", "ID_MISSION")
mission_paris_psdf["TYPE_MISSION"].unique()

c:\Users\etien\miniconda3\envs\nf26\Lib\site-packages\pyspark\pandas\utils.py:1037: PandasAPIOnSparkAdviceWarning: `to_list` loads all data into the driver's memory. It should only be used if the resulting list is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


0               Conférence
1                Formation
2                  Réunion
3    Rencontre entreprises
4            Développement
Name: TYPE_MISSION, dtype: object

In [ ]:
def clean_langue_mission(psdf, site):
    match site:
        case "BERLIN":
            map_type_mission = {
                "Geschäftstreffen": "Business Meeting",
                "Konferenz": "Conference",
                "Schulung": "Vocational Training",
                "Meeting": "Team Meeting",
                "Entwicklung": "Development",
            }
            psdf["TYPE_MISSION"] = psdf["TYPE_MISSION"].replace(map_type_mission)
        case "PARIS":
            map_type_mission = {
                "Conférence": "Conference",
                "Formation": "Vocational Training",
                "Réunion": "Team Meeting",
                "Rencontre entreprises": "Business Meeting",
                "Développement": "Development",
            }
            psdf["TYPE_MISSION"] = psdf["TYPE_MISSION"].replace(map_type_mission)

#### Test de réponse à une question 

Combien d’ingénieurs informaticiens travaillent sur le site de Paris ?

In [15]:
pers_paris_psdf = readFiles("data/BDD_BGES/BDD_BGES_PARIS", "ID_PERSONNEL")
pers_paris_sdf = pers_paris_psdf.to_spark(index_col="ID_PERSONNEL")

c:\Users\etien\miniconda3\envs\nf26\Lib\site-packages\pyspark\pandas\utils.py:1037: PandasAPIOnSparkAdviceWarning: `to_list` loads all data into the driver's memory. It should only be used if the resulting list is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


In [16]:
nb_inge_info = (
    pers_paris_sdf
    .filter(col("FONCTION_PERSONNEL") == "Ingénieur Informaticien")
    .count()
)

print(f"{nb_inge_info} ingénieurs informaticiens travaillent sur le site de Paris")

1873 ingénieurs informaticiens travaillent sur le site de Paris
